In [8]:
from PIL import Image, ImageDraw
import math

Реализовать алгоритмы закраски замкнутых областей: итеративный алгоритм с затравкой, построчный алгоритм заливки с затравкой, построчный алгоритм со списком реберных точек, построчный алгоритм со списком активных ребер.

Итеративный алгоритм заливки с затравкой

In [3]:
def alg_1(img, x, y, color):
    w, h = img.size
    pixels = img.load()
    stack = [(x, y)]
    start_color = pixels[x, y]
    if start_color == color:
        return
    pixels[x, y] = color
    while stack:
        cx, cy = stack.pop()
        for nx, ny in [(cx+1, cy), (cx-1, cy), (cx, cy+1), (cx, cy-1)]:
            if 0 <= nx < w and 0 <= ny < h and pixels[nx, ny] == start_color:
                pixels[nx, ny] = color
                stack.append((nx, ny))

In [4]:
img = Image.open('img1.PNG').convert('RGB')
alg_1(img, 23, 21, (255, 0, 0))
alg_1(img, 0, 0, (0, 255, 0))
img.show()

Построчный алгоритм заливки с затравкой

In [5]:
def alg_2(img, x, y, color):
    w, h = img.size
    pixels = img.load()
    stack = [(x, y)]
    start_color = pixels[x, y]
    if start_color == color:
        return
    while stack:
        x, y = stack.pop()
        left_x = x
        right_x = x
        while left_x > 0 and pixels[left_x - 1, y] == start_color:
            left_x -= 1
        while right_x < w - 1 and pixels[right_x + 1, y] == start_color:
            right_x += 1
        for i in range(left_x, right_x + 1):
            pixels[i, y] = color
        for i in [y - 1, y + 1]:
            if 0 <= i <= h - 1:
                j = left_x
                while j <= right_x:
                    while j <= right_x and pixels[j, i] != start_color:
                        j += 1
                    if j <= right_x and pixels[j, i] == start_color:
                        while j <= right_x and pixels[j, i] == start_color:
                            j += 1
                        stack.append((j - 1, i))

In [6]:
img = Image.open('img1.PNG').convert('RGB')
alg_2(img, 23, 21, (255, 0, 0))
alg_2(img, 0, 0, (0, 255, 0))
img.show()

Построчный алгоритм со списком реберных точек

In [25]:
def project(vertex):
    x = int((vertex[0]) * 150)
    y = int(vertex[1] * 150)
    return x, y
def dots_y(vertices, face):
    dots = [[] for i in range(501)]
    for i in range(len(face)):
        v1 = vertices[face[i]]
        v2 = vertices[face[(i + 1) % len(face)]]
        x0, y0 = project(v1)
        x1, y1 = project(v2)
        if y0 == y1: continue
        if y0 > y1: x0, y0, x1, y1 = x1, y1, x0, y0
        y = math.ceil(y0)
        dx = (x1 - x0) / (y1 - y0)
        x = x0 + dx * (y - y0)
        while y <= (y1 - 1):
            dots[250 - y].append(250 + x)
            y += 1
            x += dx
    for i in range(501):
        dots[i].sort()
    return dots
def printer(img, dots):
    w, h = img.size
    pixels = img.load()
    for y_index, row_intersections in enumerate(dots):
        if not row_intersections:
            continue
        for j in range(0, len(row_intersections) - 1, 2):
            x_start = math.ceil(row_intersections[j])
            x_end = math.floor(row_intersections[j+1])
            x_start = max(0, x_start)
            x_end = min(w - 1, x_end)
            for x in range(x_start, x_end + 1):
                pixels[x, y_index] = (255, 0, 0)
def alg_3(vertices, face):
    dots = dots_y(vertices, face)
    img = Image.new('RGB', (501, 501), color=(255, 255, 255))
    printer(img, dots)
    img.show()

In [26]:
vertices = [[-1, -1, 0], [1, -1, 0], [1, 1, 0], [-1, 1, 0]]
face = [0, 1, 2, 3]
alg_3(vertices, face)

построчный алгоритм со списком активных ребер

In [36]:
def y_list(vertices, face):
    y_table = [[] for i in range(501)]
    for i in range(len(face)):
        v1 = vertices[face[i]]
        v2 = vertices[face[(i + 1) % len(face)]]
        x0, y0 = project(v1)
        x1, y1 = project(v2)
        if y0 == y1: continue
        if y0 < y1: x0, y0, x1, y1 = x1, y1, x0, y0
        y_table[250 - y0].append([250 - y1, 250 + x0, (x1 - x0) / (y0 - y1)])
    return y_table
def cat(img, y_table):
    w, h = img.size
    pixels = img.load()
    aet = []
    for y_current in range(501):
        aet.extend(y_table[y_current])
        aet = [edge for edge in aet if y_current < edge[0]]
        aet.sort(key=lambda edge: edge[1])
        for i in range(0, len(aet) - 1, 2):
            x_start = math.ceil(aet[i][1])
            x_end = math.floor(aet[i + 1][1])
            for j in range(x_start, x_end + 1):
                pixels[j, y_current] = (255, 0, 0)
        for i in range(len(aet)):
            aet[i][1] += aet[i][2]
def alg_4(vertices, face):
    y_table = y_list(vertices, face)
    img = Image.new('RGB', (501, 501), color=(255, 255, 255))
    cat(img, y_table)
    img.show()

In [37]:
vertices = [[-1, -1, 0], [1, -1, 0], [1, 1, 0], [-1, 1, 0]]
face = [0, 1, 2, 3]
alg_4(vertices, face)